## Montar carpeta drive

In [ ]:
import os, shutil, random, glob

# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

SRC = "/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset_clasificacion"
DST = "/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset"

## Dividir dataset
- 80% entrenamiento
- 20% validación

In [ ]:

# Crear estructura
for split in ['train', 'val']:
    os.makedirs(f"{DST}/images/{split}", exist_ok=True)
    os.makedirs(f"{DST}/labels/{split}", exist_ok=True)

# Listar todas las imágenes con su .txt
images = sorted(glob.glob(f"{SRC}/*.jpg"))
random.seed(42)
random.shuffle(images)

split_idx = int(len(images) * 0.8)
train_imgs = images[:split_idx]
val_imgs = images[split_idx:]

for img_list, split in [(train_imgs, 'train'), (val_imgs, 'val')]:
    for img_path in img_list:
        base = os.path.splitext(os.path.basename(img_path))[0]
        txt_path = os.path.join(SRC, f"{base}.txt")

        shutil.copy(img_path, f"{DST}/images/{split}/")
        if os.path.exists(txt_path) and os.path.basename(txt_path) != "classes.txt":
            shutil.copy(txt_path, f"{DST}/labels/{split}/")
        else:
            # Crear .txt vacío para negative samples
            open(f"{DST}/labels/{split}/{base}.txt", 'w').close()

print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

## Crear `.yaml`
Crear archivo `.yaml` para pasarselo al modelo

In [ ]:
yaml_content = """
path: /content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset
train: images/train
val: images/val

nc: 2
names: ['bola_roja', 'linea_verde']
"""

with open(f"{DST}/dataset.yaml", 'w') as f:
    f.write(yaml_content)

## Verificar estructura creada

In [ ]:
ls "/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset"

# Entrenamiento Modelo YOLO26n
(fine-tuning)

## Importar YOLO

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

## Cargar YOLO26n

In [ ]:
# Cargar el modelo preentrenado YOLO26 nano
model = YOLO("yolo26n.pt")

## Entrenamiento

In [ ]:
# Entrenar con transfer learning sobre nuestro dataset
results = model.train(
    data="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset/dataset.yaml",
    epochs=100,
    imgsz=320,           # Resolución = la de la cámara del robot
    batch=32,            # lotes
    patience=20,         # Early stopping: para si no mejora en 20 épocas
    project="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs",
    name="tanque_yolo26n",
    cache=True,          # Cachea imágenes en RAM para ir más rápido
    device=0,            # Usar GPU
)

**CAMBIAR RUTA** -> fijarse en el output del entrenamiento.

Cada vez que entrenemos el modelo creará una carpeta nueva (tanque_yolo26n, tanque_yolo26n-2...tanque_yolo26n-n).

Adaptar rutas al modelo entrenado.


Ahora podemos comprobar los resultados:


*   Curvas de entrenamiento
*   Matriz de confusión


In [ ]:
from IPython.display import Image, display

# Mostrar curvas entrenamiento
display(Image('/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_yolo26n/results.png'))

# Mostrar matriz confusion
display(Image('/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_yolo26n/confusion_matrix.png'))

## Validar sobre el conjunto de validación

In [ ]:
# Celda 4: Validar sobre el conjunto de validación
model = YOLO("/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_yolo26n/weights/best.pt")
metrics = model.val(data="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset/dataset.yaml", imgsz=320)
print(f"mAP@0.5: {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.3f}")

## Comprobar con los vídeos grabados (emulan la situación real)

In [ ]:
# Celda 5: Probar con los vídeos grabados (genera vídeos con las cajas dibujadas)
results = model.predict(
    source="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/videos_prueba/",
    imgsz=320,
    conf=0.4,
    save=True,
    save_txt=True,
    project="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/inferencia_videos_yolo26n"
)

# Exportar modelo
Una vez listo el modelo, lo exportamos a ONNX y NCNN.

**NOTA YOLO26n:** Este modelo NO necesita NMS (end-to-end). El formato de salida
será diferente al de YOLOv8n. Hay que adaptar `yolo_inferencia.py` en la RPi.

In [ ]:
# Celda 6: Exportar a ONNX
model = YOLO("/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_yolo26n/weights/best.pt")
model.export(format="onnx", imgsz=320, simplify=True)

# El archivo best.onnx se guardará junto al best.pt en la carpeta weights/

In [ ]:
# NCNN es el formato más rápido en ARM (Raspberry Pi)
model.export(format="ncnn", imgsz=320)

## Comparar con YOLOv8n

Tras entrenar, comparar las métricas con el modelo v8n anterior:
- `tanque_v1-2/` (YOLOv8n): mAP50 = 0.951
- `tanque_yolo26n/` (YOLO26n): mAP50 = ???

Si YOLO26n iguala o mejora, usar ese. Ventajas: sin NMS, menos FLOPs, diseñado para edge.